# Códigos

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from google.colab import drive
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import cdist
import umap

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

drive.mount('/content/drive/')
os.chdir("/content/drive/MyDrive/MD")

df_densidade = pd.read_csv("yolo_features_densidade.csv")
df_tamanho = pd.read_csv("yolo_features_tamanho.csv")

RANDOM_STATE = 42
CORES = [plt.cm.tab10(i) for i in range(10)]


# 1. FUNÇÕES AUXILIARES MÍNIMAS (APENAS VISUALIZAÇÃO)
def mostrar_imagens(df, idxs, titulos, n_cols=4, suptitle=None):
    n = len(idxs)
    if n == 0:
        return
    n_rows = math.ceil(n / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, i, t in zip(axes.flat, idxs, titulos):
        if i is None:
            continue
        try:
            ax.imshow(Image.open(df.iloc[i]["file_path"]))
        except Exception:
            ax.text(0.5, 0.5, "Erro ao carregar", ha="center", va="center")
        ax.set_title(t, fontsize=9)
    if suptitle:
        fig.suptitle(suptitle, fontsize=12)
    plt.tight_layout()
    plt.show()


def scatter_2d(ax, coords, grupos, titulo):
    grupos = np.asarray(grupos)
    for i, g in enumerate(sorted(pd.unique(grupos), key=str)):
        m = grupos == g
        ax.scatter(coords[m, 0], coords[m, 1], s=20, alpha=0.8,
                   color=CORES[i % 10], label=str(g))
    ax.set_title(titulo, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(fontsize=8)


# 2. PIPELINE ENXUTO DA ATIVIDADE
def executar_atividade(df, nome_dataset, k_escolhido, altura_corte=None,
                       tsne_perp=30, umap_neigh=15, umap_min_dist=0.1, n_exemplos=4):
    df = df.reset_index(drop=True)
    y = df["class_name"].values
    classes = df["class_name"].unique()
    feature_cols = [c for c in df.columns if c.startswith("f_")]
    X_raw = df[feature_cols].values.astype(float)
    N, D = X_raw.shape

    # Pré-processamento: StandardScaler (Z-Score) + Normalização L2
    X = normalize(StandardScaler().fit_transform(X_raw))

    # ETAPA 1: CONSTRUÇÃO E DOCUMENTAÇÃO DO CONJUNTO DE DADOS
    print("=== Dados 1 ===")
    print(f"Dataset / Origem          : {nome_dataset} (yolo_features_{nome_dataset.lower()}.csv)")
    print(f"Tipo dos dados            : Imagens (representadas por vetores de características)")
    print(f"Modelo de Embeddings      : YOLO (features extraídas na camada latente)")
    print(f"Quantidade de objetos (N) : {N}")
    print(f"Dimensão do embedding (D) : {D}")
    print(f"Pré-processamento         : Padronização Z-score (StandardScaler) + Normalização L2")
    print("\nDistribuição dos rótulos originais (apenas para referência futura):")
    for c in classes:
        qtd = (y == c).sum()
        print(f"  - {c}: {qtd} amostras ({qtd / N * 100:.2f}%)")
    print("\nExemplos de caminhos dos dados originais (primeiros 5):")
    for idx_ex in range(min(5, N)):
        print(f"  [{idx_ex}] Classe: {y[idx_ex]} | Arquivo: {df.iloc[idx_ex]['file_path']}")

    print("\n=== Imagem 1 ===")
    idxs_amostra, tits_amostra = [], []
    for c in classes:
        pos = np.where(y == c)[0][:n_exemplos]
        for p in pos:
            idxs_amostra.append(p)
            tits_amostra.append(f"ID:{p} | {c}")
    mostrar_imagens(df, idxs_amostra, tits_amostra, n_cols=n_exemplos,
                    suptitle=f"Exemplos dos Dados Originais ({nome_dataset})")

    # ETAPA 2: EXPLORAÇÃO DO ESPAÇO DE EMBEDDINGS (PCA, t-SNE, UMAP)
    pca = PCA(n_components=2, random_state=RANDOM_STATE)
    X_pca = pca.fit_transform(X)
    var_pca = pca.explained_variance_ratio_ * 100

    tsne = TSNE(n_components=2, perplexity=tsne_perp, init="pca", random_state=RANDOM_STATE)
    X_tsne = tsne.fit_transform(X)

    reducer = umap.UMAP(n_components=2, n_neighbors=umap_neigh, min_dist=umap_min_dist, random_state=RANDOM_STATE)
    X_umap = reducer.fit_transform(X)

    print("\n=== Dados 2 ===")
    print("Parâmetros e Métricas da Redução de Dimensionalidade (2D):")
    print(f"  - PCA   : PC1 = {var_pca[0]:.2f}%, PC2 = {var_pca[1]:.2f}% (Variância Explicada Total = {var_pca.sum():.2f}%)")
    print(f"  - t-SNE : n_components=2, perplexity={tsne_perp}, init='pca', random_state={RANDOM_STATE}")
    print(f"  - UMAP  : n_components=2, n_neighbors={umap_neigh}, min_dist={umap_min_dist}, random_state={RANDOM_STATE}")

    print("\n=== Imagem 2 ===")
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    scatter_2d(axes[0], X_pca, y, f"PCA 2D ({var_pca.sum():.1f}% var)")
    scatter_2d(axes[1], X_tsne, y, f"t-SNE 2D (perp={tsne_perp})")
    scatter_2d(axes[2], X_umap, y, f"UMAP 2D (neigh={umap_neigh})")
    fig.suptitle(f"Exploração do Espaço de Embeddings — {nome_dataset}", fontsize=12)
    plt.tight_layout()
    plt.show()

    # ETAPAS 3 E 4: AGRUPAMENTO COM K-MEANS E ESCOLHA DE K
    k_max = max(2, int(math.floor(math.sqrt(N))))
    resultados_km = []
    for k in range(2, k_max + 1):
        km_temp = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
        sil = silhouette_score(X, km_temp.labels_)
        resultados_km.append({"K": k, "Inertia (WCSS)": km_temp.inertia_, "Silhouette": sil})
    df_km = pd.DataFrame(resultados_km)

    print("\n=== Dados 3 ===")
    print(f"Avaliação do K-Means para K = 2 até floor(sqrt({N})) = {k_max}:")
    print(df_km.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    melhor_k_sil = df_km.loc[df_km["Silhouette"].idxmax(), "K"]
    maior_sil = df_km["Silhouette"].max()
    print(f"\nPico numérico do Silhouette Score: K = {melhor_k_sil} (Silhouette = {maior_sil:.4f})")

    print("\n=== Imagem 3 ===")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(df_km["K"], df_km["Inertia (WCSS)"], marker="o")
    axes[0].set_title(f"Elbow Method (K x Inertia) — {nome_dataset}")
    axes[0].set_xlabel("K")
    axes[0].set_ylabel("Inertia (WCSS)")
    axes[0].grid(alpha=0.3)

    axes[1].plot(df_km["K"], df_km["Silhouette"], marker="o", color="tab:orange")
    axes[1].set_title(f"Silhouette Score (K x Silhouette) — {nome_dataset}")
    axes[1].set_xlabel("K")
    axes[1].set_ylabel("Silhouette Score")
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # ETAPA 5: AGRUPAMENTO HIERÁRQUICO E ANÁLISE DO DENDROGRAMA
    Z = linkage(X, method="ward", metric="euclidean")
    if altura_corte is None:
        # Define a altura de corte automaticamente para gerar 'k_escolhido' grupos
        altura_corte = (Z[-(k_escolhido - 1), 2] + Z[-k_escolhido, 2]) / 2.0

    lab_hier = fcluster(Z, t=altura_corte, criterion="distance")
    n_grupos_hier = len(np.unique(lab_hier))

    print("\n=== Dados 4 ===")
    print("Agrupamento Hierárquico Aglomerativo (Método: Ward | Métrica: Euclidiana)")
    print("Últimas 10 fusões do dendrograma (maiores distâncias):")
    ultimas = Z[-10:]
    for i, linha in enumerate(ultimas, start=len(Z) - 9):
        k_antes = N - i + 1
        print(f"  Fusão {i:3d} ({k_antes:2d} -> {k_antes-1:2d} grupos) | Distância (altura): {linha[2]:.4f} | Tam. resultante: {int(linha[3])}")
    print(f"\nAltura de corte aplicada : {altura_corte:.4f}")
    print(f"Grupos gerados pelo corte: {n_grupos_hier}")

    print("\n=== Imagem 4 ===")
    plt.figure(figsize=(12, 5))
    dendrogram(Z, truncate_mode="lastp", p=30, color_threshold=altura_corte, show_leaf_counts=True)
    plt.axhline(altura_corte, ls="--", c="k", lw=1.2, label=f"Corte = {altura_corte:.2f} ({n_grupos_hier} grupos)")
    plt.title(f"Dendrograma (Ward) — {nome_dataset}")
    plt.xlabel("Índices das Amostras (ou tamanho do nó agrupado entre parênteses)")
    plt.ylabel("Distância de Fusão (Ward)")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # ETAPA 6 E 7: DEFINIÇÃO DE K E INTERPRETAÇÃO DOS GRUPOS
    km_final = KMeans(n_clusters=k_escolhido, n_init=10, random_state=RANDOM_STATE).fit(X)
    lab_km = km_final.labels_
    centroides = km_final.cluster_centers_

    print("\n=== Dados 5 ===")
    print(f"Análise e Interpretação para K = {k_escolhido} grupos (K-Means):")
    vals, counts = np.unique(lab_km, return_counts=True)
    for c_id, cnt in zip(vals, counts):
        print(f"  - Cluster C{c_id}: {cnt} objetos ({cnt / N * 100:.2f}%)")

    print("\nComposição de classes originais dentro de cada Cluster (Contagem Absoluta):")
    print(pd.crosstab(lab_km, y, rownames=["Cluster"], colnames=["Classe"]))

    print("\nComposição de classes originais dentro de cada Cluster (Porcentagem % por linha):")
    print((pd.crosstab(lab_km, y, normalize="index") * 100).round(2))

    idxs_prot, tits_prot = [], []
    print("\nExemplos representativos (mais próximos ao centróide de cada grupo):")
    for c_id in vals:
        pos = np.where(lab_km == c_id)[0]
        dists = np.linalg.norm(X[pos] - centroides[c_id], axis=1)
        mais_prox = pos[np.argsort(dists)[:n_exemplos]]
        print(f"  Cluster C{c_id}: Índices {list(mais_prox)} | Classes: {list(y[mais_prox])}")
        for p in mais_prox:
            idxs_prot.append(p)
            tits_prot.append(f"C{c_id} | ID:{p}\n{y[p]}")

    print("\n=== Imagem 5 ===")
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    grupos_str = [f"C{c}" for c in lab_km]
    scatter_2d(axes[0], X_pca, grupos_str, f"PCA 2D — K-Means (K={k_escolhido})")
    scatter_2d(axes[1], X_tsne, grupos_str, f"t-SNE 2D — K-Means (K={k_escolhido})")
    scatter_2d(axes[2], X_umap, grupos_str, f"UMAP 2D — K-Means (K={k_escolhido})")
    fig.suptitle(f"Visualização dos Grupos Encontrados (K={k_escolhido}) — {nome_dataset}", fontsize=12)
    plt.tight_layout()
    plt.show()

    mostrar_imagens(df, idxs_prot, tits_prot, n_cols=n_exemplos,
                    suptitle=f"Protótipos de Cada Cluster (K={k_escolhido}) — {nome_dataset}")

    # ETAPAS 8 E 9: GRUPOS MUITO DIFERENTES (DISTANTES) E GRUPOS PRÓXIMOS
    dist_matrix = cdist(centroides, centroides, metric="euclidean")
    df_dist = pd.DataFrame(dist_matrix,
                           index=[f"C{i}" for i in vals],
                           columns=[f"C{i}" for i in vals])

    tri_u = np.triu_indices(len(vals), k=1)
    dists_pares = dist_matrix[tri_u]
    idx_max = np.argmax(dists_pares)
    idx_min = np.argmin(dists_pares)

    c_dist_1, c_dist_2 = tri_u[0][idx_max], tri_u[1][idx_max]
    c_prox_1, c_prox_2 = tri_u[0][idx_min], tri_u[1][idx_min]

    print("\n=== Dados 6 ===")
    print("Matriz de Distância Euclidiana entre os Centróides dos Clusters:")
    print(df_dist.round(4))
    print(f"\nPar de grupos MAIS DISTANTES (Diferentes) : C{c_dist_1} e C{c_dist_2} (Distância = {dists_pares[idx_max]:.4f})")
    print(f"Par de grupos MAIS PRÓXIMOS (Semelhantes) : C{c_prox_1} e C{c_prox_2} (Distância = {dists_pares[idx_min]:.4f})")

    # Seleciona exemplos dos grupos mais distantes e dos grupos mais próximos
    idxs_comp, tits_comp = [], []
    for c_id, tag in [(c_dist_1, "Distante"), (c_dist_2, "Distante")]:
        pos = np.where(lab_km == c_id)[0]
        escolhidos = pos[np.argsort(np.linalg.norm(X[pos] - centroides[c_id], axis=1))[:n_exemplos]]
        for p in escolhidos:
            idxs_comp.append(p)
            tits_comp.append(f"[{tag}] C{c_id}\nID:{p} | {y[p]}")

    # Se K > 2, adiciona também o par mais próximo na visualização; se K == 2, o par distante e próximo são o mesmo
    if len(vals) > 2:
        for c_id, tag in [(c_prox_1, "Próximo"), (c_prox_2, "Próximo")]:
            pos = np.where(lab_km == c_id)[0]
            escolhidos = pos[np.argsort(np.linalg.norm(X[pos] - centroides[c_id], axis=1))[:n_exemplos]]
            for p in escolhidos:
                idxs_comp.append(p)
                tits_comp.append(f"[{tag}] C{c_id}\nID:{p} | {y[p]}")

    print("\n=== Imagem 6 ===")
    mostrar_imagens(df, idxs_comp, tits_comp, n_cols=n_exemplos,
                    suptitle=f"Comparação: Grupos Mais Distantes (C{c_dist_1} vs C{c_dist_2}) e Mais Próximos (C{c_prox_1} vs C{c_prox_2})")

    # ETAPA 10: CASOS DIFÍCEIS E PONTOS ATÍPICOS
    sil_samples = silhouette_samples(X, lab_km)
    dist_ao_centroide = np.array([np.linalg.norm(X[i] - centroides[lab_km[i]]) for i in range(N)])

    # 1. Pontos de fronteira / sobreposição (menor silhouette)
    piores_sil = np.argsort(sil_samples)[:n_exemplos]
    # 2. Pontos atípicos / isolados (mais distantes do próprio centróide)
    mais_longe = np.argsort(-dist_ao_centroide)[:n_exemplos]

    print("\n=== Dados 7 ===")
    print(f"Total de objetos com Silhouette negativo (possível grupo incorreto/fronteira): {(sil_samples < 0).sum()} de {N} ({(sil_samples < 0).mean()*100:.2f}%)")
    print(f"\nTop {n_exemplos} casos ambíguos / fronteira entre grupos (Menor Silhouette Individual):")
    for p in piores_sil:
        print(f"  ID:{p:4d} | Cluster: C{lab_km[p]} | Classe Real: {y[p]:12s} | Silhouette: {sil_samples[p]:+.4f} | arquivo: {df.iloc[p]['file_path']}")

    print(f"\nTop {n_exemplos} pontos atípicos / isolados (Maior distância ao próprio centróide):")
    for p in mais_longe:
        print(f"  ID:{p:4d} | Cluster: C{lab_km[p]} | Classe Real: {y[p]:12s} | Distância : {dist_ao_centroide[p]:.4f} | arquivo: {df.iloc[p]['file_path']}")

    print("\n=== Imagem 7 ===")
    idxs_dif, tits_dif = [], []
    for p in piores_sil:
        idxs_dif.append(p)
        tits_dif.append(f"[Fronteira] ID:{p}\nC{lab_km[p]} | {y[p]} (sil={sil_samples[p]:.2f})")
    for p in mais_longe:
        idxs_dif.append(p)
        tits_dif.append(f"[Atípico] ID:{p}\nC{lab_km[p]} | {y[p]} (d={dist_ao_centroide[p]:.2f})")

    mostrar_imagens(df, idxs_dif, tits_dif, n_cols=n_exemplos,
                    suptitle=f"Casos Difíceis: Fronteira (1ª linha) e Atípicos/Outliers (2ª linha) — {nome_dataset}")

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


# Resultado

In [ ]:
executar_atividade(df_densidade, nome_dataset="Densidade", k_escolhido=2)

Output hidden; open in https://colab.research.google.com to view.

In [7]:
executar_atividade(df_tamanho, nome_dataset="Tamanho", k_escolhido=3)

Output hidden; open in https://colab.research.google.com to view.